# 04 - Validacion de Calidad de Datos
## Compuerta de Calidad (Quality Gate) - Capa Bronce

---

### Objetivo de este notebook

Este notebook implementa una **compuerta de calidad** (quality gate) que verifica
la integridad y coherencia de los datos en la capa Bronce **antes** de permitir
su transformacion hacia capas superiores.

### Por que una compuerta de calidad?

En ingenieria de datos, es una practica critica validar los datos en cada etapa
del pipeline para evitar que datos corruptos, incompletos o incoherentes se
propaguen a las capas de consumo donde podrian generar:
- Reportes erroneos
- Decisiones basadas en datos incorrectos
- Fallos en procesos downstream

### Reglas implementadas

Se definen **6 reglas de validacion** con dos niveles de severidad:

| # | Regla | Severidad | Descripcion |
|---|-------|-----------|-------------|
| 1 | Completitud de compuestos | CRITICA | Los 15 medicamentos deben estar presentes |
| 2 | Registros minimos de bioactividad | CRITICA | Se esperan al menos 10,000 registros |
| 3 | CIDs consistentes entre tablas | CRITICA | Todos los CID de bioactividad deben existir en compuestos |
| 4 | Nulidad de campos clave | ADVERTENCIA | CID y activity_outcome no deben ser nulos en >5% |
| 5 | Rango de valores numericos | ADVERTENCIA | MolecularWeight debe estar entre 50 y 1500 Da |
| 6 | Duplicados exactos | ADVERTENCIA | No mas de 1% de duplicados exactos en bioactividad |

### Comportamiento de la compuerta

- Si **alguna regla CRITICA falla**, el pipeline se detiene con una excepcion
- Si **solo fallan reglas de ADVERTENCIA**, el pipeline continua con un aviso

---
## 1. Importacion de Librerias y Configuracion

In [ ]:
from pyspark.sql import functions as F  # Funciones de transformacion de Spark SQL
from datetime import datetime            # Para registrar la hora de validacion

In [ ]:
# Detectar catalogo activo de Unity Catalog
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]
ESQUEMA_BRONCE = "pubchem_bronce"  # Esquema de la capa Bronce a validar

# Mostrar configuracion y hora de inicio
print(f"Catalogo: {CATALOGO}")
print(f"Esquema:  {ESQUEMA_BRONCE}")
print(f"Inicio de validacion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. Carga de Tablas y Funcion de Registro

Cargamos las tablas de la capa Bronce y definimos una funcion auxiliar
para registrar los resultados de cada regla de validacion de forma uniforme.

In [ ]:
# Cargar las tablas de la capa Bronce
df_compuestos = spark.table(f"{ESQUEMA_BRONCE}.propiedades_compuestos")  # Tabla de propiedades
df_bioactividad = spark.table(f"{ESQUEMA_BRONCE}.bioactividad")          # Tabla de bioactividad

# Obtener conteos totales que se usaran en las validaciones
total_compuestos = df_compuestos.count()
total_bioactividad = df_bioactividad.count()

print(f"Registros en propiedades_compuestos: {total_compuestos}")
print(f"Registros en bioactividad:           {total_bioactividad:,}")

In [ ]:
# Lista para acumular los resultados de todas las reglas de validacion
resultados_validacion = []


def registrar_resultado(numero, nombre, severidad, paso, detalle):
    """
    Registra el resultado de una regla de validacion y lo muestra en consola.

    Esta funcion centraliza el registro de resultados para garantizar
    un formato uniforme y facilitar el resumen final.

    Parametros
    ----------
    numero : int
        Numero identificador de la regla.
    nombre : str
        Nombre descriptivo de la regla.
    severidad : str
        Nivel de severidad ('CRITICA' o 'ADVERTENCIA').
    paso : bool
        True si la regla fue aprobada, False si fallo.
    detalle : str
        Descripcion detallada del resultado.
    """
    # Determinar el estado textual e icono segun el resultado
    estado = "APROBADA" if paso else "FALLIDA"
    icono = "[OK]" if paso else "[FALLO]"

    # Almacenar resultado en la lista acumuladora
    resultados_validacion.append({
        "regla": numero,
        "nombre": nombre,
        "severidad": severidad,
        "estado": estado,
        "detalle": detalle
    })

    # Mostrar resultado en consola con formato legible
    print(f"  Regla {numero} {icono} [{severidad}] {nombre}")
    print(f"    -> {detalle}")
    print()

---
## 3. Regla 1: Completitud de Compuestos (CRITICA)

Verifica que los **15 medicamentos** esperados esten presentes en la tabla
de propiedades de compuestos. Si faltan compuestos, la ingesta fue incompleta
y el pipeline no debe continuar.

In [ ]:
# Numero esperado de compuestos segun la lista de medicamentos del notebook 01
COMPUESTOS_ESPERADOS = 15

# Contar compuestos unicos por CID en la tabla de propiedades
compuestos_unicos = df_compuestos.select("CID").distinct().count()

# Evaluar si la regla se cumple
paso_r1 = compuestos_unicos >= COMPUESTOS_ESPERADOS

# Registrar resultado
registrar_resultado(
    1,
    "Completitud de compuestos",
    "CRITICA",
    paso_r1,
    f"Compuestos unicos encontrados: {compuestos_unicos} (esperados: {COMPUESTOS_ESPERADOS})"
)

---
## 4. Regla 2: Registros Minimos de Bioactividad (CRITICA)

La tabla de bioactividad debe contener al menos **10,000 registros** para
considerarse valida. Un numero menor indicaria que la extraccion del API
fallo parcialmente o que el API retorno datos incompletos.

In [ ]:
# Umbral minimo de registros esperados en la tabla de bioactividad
MINIMO_REGISTROS = 10000

# Evaluar si la cantidad de registros supera el umbral
paso_r2 = total_bioactividad >= MINIMO_REGISTROS

# Registrar resultado
registrar_resultado(
    2,
    "Registros minimos de bioactividad",
    "CRITICA",
    paso_r2,
    f"Registros encontrados: {total_bioactividad:,} (minimo esperado: {MINIMO_REGISTROS:,})"
)

---
## 5. Regla 3: Consistencia de CIDs entre Tablas (CRITICA)

Todos los CID (identificadores de compuesto) presentes en la tabla de bioactividad
deben existir en la tabla de propiedades de compuestos. Los CIDs "huerfanos"
(presentes en bioactividad pero no en propiedades) indicarian un problema de
integridad referencial que afectaria los JOINs en capas superiores.

**Nota tecnica:** En propiedades la columna se llama `CID` (mayusculas, nombre
original de PubChem), mientras que en bioactividad se llama `cid` (minusculas,
renombrada en la capa Bronce).

In [ ]:
# Obtener el conjunto de CIDs unicos de la tabla de propiedades
# Se convierten a string para comparacion uniforme
cids_compuestos = set(
    str(fila["CID"]) for fila in df_compuestos.select("CID").distinct().collect()
)

# Obtener el conjunto de CIDs unicos de la tabla de bioactividad
# Se filtran nulos para evitar errores en la comparacion
cids_bioactividad = set(
    str(fila["cid"]) for fila in df_bioactividad.select("cid").distinct().collect()
    if fila["cid"] is not None
)

# Calcular CIDs huerfanos: presentes en bioactividad pero no en propiedades
cids_huerfanos = cids_bioactividad - cids_compuestos

# La regla se cumple si no hay CIDs huerfanos
paso_r3 = len(cids_huerfanos) == 0

# Construir detalle del resultado
detalle_r3 = f"CIDs en bioactividad: {len(cids_bioactividad)}, CIDs en compuestos: {len(cids_compuestos)}"
if not paso_r3:
    detalle_r3 += f", CIDs huerfanos: {cids_huerfanos}"

# Registrar resultado
registrar_resultado(
    3,
    "Consistencia de CIDs entre tablas",
    "CRITICA",
    paso_r3,
    detalle_r3
)

---
## 6. Regla 4: Nulidad de Campos Clave (ADVERTENCIA)

Los campos `cid` y `activity_outcome` son esenciales para el modelo analitico:
- `cid` se usa como llave de JOIN con la tabla de propiedades
- `activity_outcome` es el campo de resultado principal del bioensayo

Si mas del 5% de registros tienen estos campos nulos o vacios,
se genera una advertencia (pero no se detiene el pipeline).

In [ ]:
# Umbral maximo de nulidad permitido (porcentaje)
UMBRAL_NULIDAD = 5.0

# Campos clave a evaluar en la tabla de bioactividad
campos_clave = ["cid", "activity_outcome"]
detalles_nulidad = []  # Para acumular detalles de cada campo
paso_r4 = True         # Asumimos que pasa hasta encontrar un fallo

# Evaluar nulidad de cada campo clave
for campo in campos_clave:
    # Contar registros donde el campo es nulo o cadena vacia
    nulos = df_bioactividad.filter(
        (F.col(campo).isNull()) | (F.col(campo) == "")
    ).count()

    # Calcular porcentaje de nulidad
    porcentaje_nulos = round(nulos * 100 / total_bioactividad, 2)
    detalles_nulidad.append(f"{campo}: {porcentaje_nulos}% nulos ({nulos:,} registros)")

    # Si algun campo supera el umbral, la regla falla
    if porcentaje_nulos > UMBRAL_NULIDAD:
        paso_r4 = False

# Registrar resultado
registrar_resultado(
    4,
    "Nulidad de campos clave",
    "ADVERTENCIA",
    paso_r4,
    f"Umbral: {UMBRAL_NULIDAD}% | " + " | ".join(detalles_nulidad)
)

---
## 7. Regla 5: Rango de Valores Numericos (ADVERTENCIA)

El peso molecular (`MolecularWeight`) de un compuesto farmacologicamente
relevante debe estar entre **50 y 1500 Daltons**. Valores fuera de este rango
podrian indicar errores en los datos o compuestos no relevantes.

Referencia: Los farmacos tipicos tienen pesos moleculares entre 150 y 500 Da,
pero se usa un rango amplio para no excluir moleculas biologicas mas grandes.

In [ ]:
# Limites del rango valido de peso molecular en Daltons
MW_MIN = 50    # Minimo: moleculas muy pequenas no suelen ser farmacos
MW_MAX = 1500  # Maximo: moleculas muy grandes raramente son biodisponibles por via oral

# Contar compuestos con peso molecular fuera del rango valido
fuera_rango = df_compuestos.filter(
    (F.col("MolecularWeight").cast("double") < MW_MIN) |
    (F.col("MolecularWeight").cast("double") > MW_MAX)
).count()

# La regla se cumple si ningun compuesto esta fuera de rango
paso_r5 = fuera_rango == 0

# Registrar resultado
registrar_resultado(
    5,
    "Rango de peso molecular",
    "ADVERTENCIA",
    paso_r5,
    f"Compuestos fuera de rango [{MW_MIN}-{MW_MAX} Da]: {fuera_rango} de {total_compuestos}"
)

---
## 8. Regla 6: Duplicados Exactos en Bioactividad (ADVERTENCIA)

Verifica que no exista mas del **1%** de filas completamente duplicadas
en la tabla de bioactividad. Los duplicados pueden originarse por:
- Multiples ingestas del mismo periodo
- Errores en el API de PubChem
- Problemas en la escritura a Delta

Se excluyen las columnas de trazabilidad (que empiezan con `_`) de la
comparacion, ya que son diferentes entre ejecuciones por diseno.

In [ ]:
# Umbral maximo de duplicados permitido (porcentaje)
UMBRAL_DUPLICADOS = 1.0

# Seleccionar solo columnas de negocio (excluir columnas de trazabilidad)
columnas_negocio = [c for c in df_bioactividad.columns if not c.startswith("_")]

# Contar registros distintos usando solo columnas de negocio
registros_distintos = df_bioactividad.select(columnas_negocio).distinct().count()

# Calcular cantidad y porcentaje de duplicados
duplicados = total_bioactividad - registros_distintos
porcentaje_duplicados = round(duplicados * 100 / total_bioactividad, 2)

# La regla se cumple si el porcentaje de duplicados no supera el umbral
paso_r6 = porcentaje_duplicados <= UMBRAL_DUPLICADOS

# Registrar resultado
registrar_resultado(
    6,
    "Duplicados exactos en bioactividad",
    "ADVERTENCIA",
    paso_r6,
    f"Duplicados: {duplicados:,} ({porcentaje_duplicados}%) | Umbral: {UMBRAL_DUPLICADOS}%"
)

---
## 9. Resumen de Validacion

Se presenta un resumen consolidado de todas las reglas de validacion ejecutadas,
con el conteo de reglas aprobadas vs fallidas por nivel de severidad.

In [ ]:
# Encabezado del resumen
print("=" * 70)
print("         RESUMEN DE VALIDACION DE CALIDAD - CAPA BRONCE")
print("=" * 70)
print()

# Listas para acumular reglas fallidas por severidad
criticas_fallidas = []
advertencias_fallidas = []

# Iterar sobre los resultados y clasificar las reglas fallidas
for resultado in resultados_validacion:
    marca = "[OK]" if resultado["estado"] == "APROBADA" else "[FALLO]"
    print(f"  {marca}  Regla {resultado['regla']}: {resultado['nombre']} ({resultado['severidad']})")

    # Clasificar reglas fallidas por severidad
    if resultado["estado"] == "FALLIDA":
        if resultado["severidad"] == "CRITICA":
            criticas_fallidas.append(resultado)
        else:
            advertencias_fallidas.append(resultado)

# Calcular totales
aprobadas = sum(1 for r in resultados_validacion if r["estado"] == "APROBADA")
total_reglas = len(resultados_validacion)

# Mostrar resumen numerico
print()
print(f"  Resultado: {aprobadas}/{total_reglas} reglas aprobadas")
print(f"  Reglas criticas fallidas:    {len(criticas_fallidas)}")
print(f"  Advertencias:                {len(advertencias_fallidas)}")
print("=" * 70)

---
## 10. Compuerta de Calidad (Quality Gate)

Este es el punto de decision critico del pipeline:

- Si **alguna regla CRITICA fallo**, se lanza una excepcion que detiene el pipeline
  para evitar propagar datos corruptos a las capas Plata y Oro
- Si **solo fallaron reglas de ADVERTENCIA**, el pipeline continua pero se emite
  un mensaje de alerta para que el equipo investigue
- Si **todas las reglas pasaron**, el pipeline continua normalmente

In [ ]:
# Evaluar la compuerta de calidad
if criticas_fallidas:
    # CASO 1: Reglas criticas fallidas -> DETENER el pipeline
    # Construir mensaje de error detallado con las reglas que fallaron
    mensajes = [
        f"Regla {r['regla']}: {r['nombre']} - {r['detalle']}"
        for r in criticas_fallidas
    ]
    error_detalle = "\n  ".join(mensajes)

    # Lanzar excepcion para detener la ejecucion del pipeline
    raise Exception(
        f"COMPUERTA DE CALIDAD FALLIDA: {len(criticas_fallidas)} regla(s) critica(s) no cumplida(s).\n"
        f"  {error_detalle}\n\n"
        f"El pipeline se detiene para evitar propagar datos corruptos a capas superiores."
    )

if advertencias_fallidas:
    # CASO 2: Solo advertencias -> CONTINUAR con alerta
    print("ADVERTENCIA: Algunas reglas no criticas fallaron. Revise los detalles arriba.")
    print("El pipeline continua, pero se recomienda investigar las advertencias.")
else:
    # CASO 3: Todas las reglas aprobadas -> CONTINUAR normalmente
    print("Todas las reglas de calidad fueron aprobadas. El pipeline puede continuar.")